In [1]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import BertTokenizer, BertModel

# === Paths ===
BASE_DIR   = Path(r"D:\Shikha\bias_project")
OUTPUT_DIR = BASE_DIR / "outputs" / "embeddings" / "bert"
WORD_LISTS = BASE_DIR / "word_lists" / "caliskan_2017"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === Load BERT ===
print("Loading BERT model and tokenizer...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model     = BertModel.from_pretrained('bert-base-uncased')
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = model.to(device)
print(f"Model loaded on : {device}")
print(f"BERT hidden size: {model.config.hidden_size}")

# === Load WEAT 6 word lists ===
def load_word_list(filepath):
    with open(filepath, 'r') as f:
        return [w.strip().lower() for w in f.readlines() if w.strip()]

career_terms  = load_word_list(WORD_LISTS / "career_terms_weat6.txt")
family_terms  = load_word_list(WORD_LISTS / "family_terms_weat6.txt")
male_names    = load_word_list(WORD_LISTS / "male_names_weat6.txt")
female_names  = load_word_list(WORD_LISTS / "female_names_weat6.txt")

print(f"\nWEAT 6 word lists loaded:")
print(f"Career : {career_terms}")
print(f"Family : {family_terms}")
print(f"Male   : {male_names}")
print(f"Female : {female_names}")

d:\Shikha\bias_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading BERT model and tokenizer...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6302.60it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on : cpu
BERT hidden size: 768

WEAT 6 word lists loaded:
Career : ['executive', 'management', 'professional', 'corporation', 'salary', 'office', 'business', 'career']
Family : ['home', 'parents', 'children', 'family', 'cousins', 'marriage', 'wedding', 'relatives']
Male   : ['john', 'paul', 'mike', 'kevin', 'steve', 'greg', 'jeff', 'bill']
Female : ['amy', 'joan', 'lisa', 'sarah', 'diana', 'kate', 'ann', 'donna']


In [2]:
# === Mean-pooled BERT embedding (matches RoBERTa for clean 2x2 comparison) ===
def get_bert_embedding(text, tokenizer, model, device):
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=64, padding=True
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    token_embeddings = outputs.last_hidden_state.squeeze(0)
    attention_mask = inputs['attention_mask'].squeeze(0).unsqueeze(-1)
    summed = (token_embeddings * attention_mask).sum(0)
    counts = attention_mask.sum(0)
    return (summed / counts).cpu().numpy()

def cosine_sim(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0.0
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def make_sentence(word):
    return f"This is {word}."

# Load Garg pairs (better coverage than names — consistent with W2V/RoBERTa)
GARG_LISTS = BASE_DIR / "word_lists" / "garg_2018"
male_pairs   = load_word_list(GARG_LISTS / "male_pairs.txt")
female_pairs = load_word_list(GARG_LISTS / "female_pairs.txt")

def seat_score(target_A_embs, target_B_embs, attr_C_embs, attr_D_embs):
    def mean_cos(emb, attr_embs):
        return np.mean([cosine_sim(emb, a) for a in attr_embs.values()])
    def assoc(emb, C, D):
        return mean_cos(emb, C) - mean_cos(emb, D)
    a = [assoc(e, attr_C_embs, attr_D_embs) for e in target_A_embs.values()]
    b = [assoc(e, attr_C_embs, attr_D_embs) for e in target_B_embs.values()]
    return np.mean(a) - np.mean(b)

# Global BERT SEAT baseline (mean-pooled, template sentences)
career_embs = {w: get_bert_embedding(make_sentence(w), tokenizer, model, device) for w in career_terms}
family_embs = {w: get_bert_embedding(make_sentence(w), tokenizer, model, device) for w in family_terms}
male_embs   = {w: get_bert_embedding(make_sentence(w), tokenizer, model, device) for w in male_names}
female_embs = {w: get_bert_embedding(make_sentence(w), tokenizer, model, device) for w in female_names}

global_seat = seat_score(male_embs, female_embs, career_embs, family_embs)
print(f"Global BERT SEAT (mean-pooled, WEAT6): {global_seat:.4f}")
print(f"Embedding dim: {list(career_embs.values())[0].shape}")

Global BERT SEAT (mean-pooled, WEAT6): 0.0164
Embedding dim: (768,)


In [3]:
def seat_score(target_A_embs, target_B_embs, attr_C_embs, attr_D_embs):
    """
    SEAT score: measures association of target A vs B with attributes C vs D
    target_A = male names, target_B = female names
    attr_C = career, attr_D = family
    Positive = male names more associated with career (gender bias)
    """
    def mean_cos(target_emb, attr_embs):
        sims = [cosine_sim(target_emb, a) for a in attr_embs.values()]
        return np.mean(sims)

    def association(target_emb, attr_C, attr_D):
        return mean_cos(target_emb, attr_C) - mean_cos(target_emb, attr_D)

    male_assoc   = [association(e, attr_C_embs, attr_D_embs) 
                    for e in target_A_embs.values()]
    female_assoc = [association(e, attr_C_embs, attr_D_embs) 
                    for e in target_B_embs.values()]

    return np.mean(male_assoc) - np.mean(female_assoc)

# === Global SEAT score (no corpus — pure BERT bias) ===
global_seat = seat_score(male_embs, female_embs, career_embs, family_embs)
print(f"=== Global SEAT Score (pretrained BERT) ===")
print(f"Score     : {global_seat:.4f}")
print(f"Direction : {'male→career' if global_seat > 0 else 'female→career'}")
print(f"\nThis reflects bias in BERT pretrained on Wikipedia + BookCorpus.")

=== Global SEAT Score (pretrained BERT) ===
Score     : 0.0164
Direction : male→career

This reflects bias in BERT pretrained on Wikipedia + BookCorpus.


In [4]:
import random
random.seed(42)
SENTENCES_PER_WORD = 5
CORPUS_DIR = BASE_DIR / "corpus"
EXCLUDE = {'GU', 'PR', 'VI', 'AS', 'MP'}

# Load corpora
cr_df       = pd.read_csv(CORPUS_DIR / "congressional_record" / "2024-01" / "speeches.csv")

ca_df       = pd.read_csv(CORPUS_DIR / "chronicling_america" / "1920-1922" / "articles_full_ocr.csv")
ca_df       = ca_df.drop_duplicates('item_id')
ca_df       = ca_df[ca_df['text_length'] >= 2000].reset_index(drop=True)
print(f"Chronicling America 1920s: {len(ca_df)} pages")

ca_1940_df  = pd.read_csv(CORPUS_DIR / "chronicling_america" / "articles_1940.csv")
ca_1940_df  = ca_1940_df[ca_1940_df['text_length'] >= 2000].reset_index(drop=True)
print(f"Chronicling America 1940: {len(ca_1940_df)} pages")

ca_1960_df  = pd.read_csv(CORPUS_DIR / "chronicling_america" / "articles_1960.csv")
ca_1960_df  = ca_1960_df[ca_1960_df['text_length'] >= 2000].reset_index(drop=True)
print(f"Chronicling America 1960: {len(ca_1960_df)} pages")

gd_2015_df  = pd.read_csv(CORPUS_DIR / "gdelt" / "2015-03" / "gdelt_articles_with_text.csv",
                           usecols=['state', 'fetch_status', 'text'])
gd_2015_df  = gd_2015_df[gd_2015_df['fetch_status'] == 'success']
gd_2015_df  = gd_2015_df[gd_2015_df['text'].notna() & (gd_2015_df['text'] != '')]
print(f"GDELT 2015: {len(gd_2015_df)} articles")

gd_df       = pd.read_csv(CORPUS_DIR / "gdelt" / "2024-01" / "gdelt_articles_with_text.csv",
                           usecols=['state', 'text'])
gd_df       = gd_df[gd_df['text'].notna() & (gd_df['text'] != '')]
print(f"GDELT 2024: {len(gd_df)} articles")

rd_posts    = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "posts.csv")
rd_comments = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "comments.csv")

# Build per-(state, period) sentence lists
state_period_sentences = {}
def add_sentences(state_abbr, period, text):
    if not isinstance(text, str) or len(text.strip()) < 20:
        return
    sents = [s.strip() for s in text.replace('\n', ' ').split('.') if len(s.strip()) > 20]
    state_period_sentences.setdefault((state_abbr, period), []).extend(sents)

# hist_1920s
for _, row in ca_df.iterrows():
    add_sentences(str(row['state_abbr']), 'hist_1920s', str(row.get('text','')))

# hist_1940
for _, row in ca_1940_df.iterrows():
    add_sentences(str(row['state_abbr']), 'hist_1940', str(row.get('text','')))

# hist_1960
for _, row in ca_1960_df.iterrows():
    add_sentences(str(row['state_abbr']), 'hist_1960', str(row.get('text','')))

# modern_2015
for _, row in gd_2015_df.iterrows():
    add_sentences(str(row['state']), 'modern_2015', str(row.get('text','')))

# modern_2020
for _, row in rd_posts.iterrows():
    add_sentences(str(row['state_abbr']), 'modern_2020', str(row.get('title',''))+'. '+str(row.get('selftext','')))
for _, row in rd_comments.iterrows():
    add_sentences(str(row['state_abbr']), 'modern_2020', str(row.get('body','')))

# modern_2024
for _, row in cr_df.iterrows():
    add_sentences(str(row['state']), 'modern_2024', str(row.get('text','')))
for _, row in gd_df.iterrows():
    add_sentences(str(row['state']), 'modern_2024', str(row.get('text','')))

print(f"\nState-period cells: {len(state_period_sentences)}")

# === Score per state-period cell ===
def get_contextual_embedding(word, sentences, n=5):
    matching = [s for s in sentences if word.lower() in s.lower()]
    if not matching:
        matching = [f"This is {word}."]
    sampled = random.sample(matching, min(n, len(matching)))
    embs = [get_bert_embedding(s[:256], tokenizer, model, device) for s in sampled]
    return np.mean(embs, axis=0)

results_bert = []
print(f"\n{'State':<7} {'Period':<13} {'Sents':>6} {'SEAT':>9} {'Direction'}")
print("-" * 50)

for (state_abbr, period) in sorted(state_period_sentences.keys()):
    if state_abbr in EXCLUDE:
        continue
    sentences = state_period_sentences[(state_abbr, period)]
    if len(sentences) < 20:
        continue
    c  = {w: get_contextual_embedding(w, sentences, SENTENCES_PER_WORD) for w in career_terms}
    f  = {w: get_contextual_embedding(w, sentences, SENTENCES_PER_WORD) for w in family_terms}
    m  = {w: get_contextual_embedding(w, sentences, SENTENCES_PER_WORD) for w in male_pairs}
    fe = {w: get_contextual_embedding(w, sentences, SENTENCES_PER_WORD) for w in female_pairs}
    score = seat_score(m, fe, c, f)
    direction = "male->career" if score > 0 else "female->career"
    print(f"{state_abbr:<7} {period:<13} {len(sentences):>6} {score:>9.4f} {direction}")
    results_bert.append({
        'state': state_abbr, 'period': period,
        'n_sentences': len(sentences),
        'seat_career_family': round(score, 4),
        'method': 'bert_seat_meanpool'
    })

print(f"\nCells scored: {len(results_bert)}")

Chronicling America 1920s: 300 pages
Chronicling America 1940: 450 pages
Chronicling America 1960: 450 pages
GDELT 2015: 609 articles
GDELT 2024: 702 articles

State-period cells: 171

State   Period         Sents      SEAT Direction
--------------------------------------------------
AK      modern_2020      283    0.0017 male->career
AK      modern_2024     1375    0.0082 male->career
AL      modern_2015      177   -0.0028 female->career
AL      modern_2020      271   -0.0163 female->career
AL      modern_2024      470   -0.0090 female->career
AR      modern_2015       93   -0.0237 female->career
AR      modern_2020      383    0.0258 male->career
AR      modern_2024      307    0.0009 male->career
AZ      modern_2015      205    0.0418 male->career
AZ      modern_2020      275   -0.0088 female->career
AZ      modern_2024      790    0.0389 male->career
CA      hist_1920s      2767    0.0019 male->career
CA      hist_1940      11807    0.0173 male->career
CA      hist_1960       7044 

In [5]:
bert_df = pd.DataFrame(results_bert).sort_values(['period', 'state'])

OUT_BERT = OUTPUT_DIR / "seat_scores_bert_v2.csv"
bert_df.to_csv(OUT_BERT, index=False)

print("=== BERT SEAT Summary (mean-pooled, period-split) ===")
print(f"Total cells scored : {len(bert_df)}")
print(f"\nBy period:")
for p in ['hist_1920s', 'modern_2020', 'modern_2024']:
    sub = bert_df[bert_df['period'] == p]
    if len(sub) > 0:
        male = (sub['seat_career_family'] > 0).sum()
        female = (sub['seat_career_family'] < 0).sum()
        print(f"  {p:<13}: {len(sub):>3} cells | {male} male->career, {female} female->career | "
              f"range [{sub['seat_career_family'].min():.4f}, {sub['seat_career_family'].max():.4f}]")

print(f"\nGlobal baseline: {global_seat:.4f}")
print(f"Saved to: {OUT_BERT}")

=== BERT SEAT Summary (mean-pooled, period-split) ===
Total cells scored : 167

By period:
  hist_1920s   :   9 cells | 9 male->career, 0 female->career | range [0.0008, 0.0211]
  modern_2020  :  50 cells | 32 male->career, 18 female->career | range [-0.0319, 0.0507]
  modern_2024  :  49 cells | 38 male->career, 11 female->career | range [-0.0567, 0.0432]

Global baseline: 0.0164
Saved to: D:\Shikha\bias_project\outputs\embeddings\bert\seat_scores_bert_v2.csv


In [6]:
# Save BERT SEAT scores
scores_bert_df = pd.DataFrame(results_bert).sort_values('state')

OUTPUT_BERT = OUTPUT_DIR / "seat_scores_bert.csv"
scores_bert_df.to_csv(OUTPUT_BERT, index=False)

print("=== SEAT Scores — BERT ===\n")
print(f"{'State':<8} {'Period':<13} {'SEAT Score':>12} {'Bias Direction'}")
print("-" * 50)
for _, row in scores_bert_df.iterrows():
    direction = "male→career" if row['seat_career_family'] > 0 else "female→career"
    print(f"{row['state']:<8} {row['period']:<13} {row['seat_career_family']:>12.4f} {direction}")

print(f"\nCells scored          : {len(scores_bert_df)}")
print(f"Global BERT baseline  : 0.0063")
print(f"State range           : {scores_bert_df['seat_career_family'].min():.4f} to {scores_bert_df['seat_career_family'].max():.4f}")
print(f"Male→career cells     : {(scores_bert_df['seat_career_family'] > 0).sum()}")
print(f"Female→career cells   : {(scores_bert_df['seat_career_family'] < 0).sum()}")
print(f"Saved to              : {OUTPUT_BERT}")

=== SEAT Scores — BERT ===

State    Period          SEAT Score Bias Direction
--------------------------------------------------
AK       modern_2020         0.0017 male→career
AK       modern_2024         0.0082 male→career
AL       modern_2015        -0.0028 female→career
AL       modern_2020        -0.0163 female→career
AL       modern_2024        -0.0090 female→career
AR       modern_2015        -0.0237 female→career
AR       modern_2020         0.0258 male→career
AR       modern_2024         0.0009 male→career
AZ       modern_2015         0.0418 male→career
AZ       modern_2020        -0.0088 female→career
AZ       modern_2024         0.0389 male→career
CA       hist_1920s          0.0019 male→career
CA       hist_1940           0.0173 male→career
CA       hist_1960           0.0057 male→career
CA       modern_2015         0.0346 male→career
CA       modern_2020         0.0137 male→career
CA       modern_2024         0.0077 male→career
CO       modern_2015         0.0043 male→car